# **MODEL EXPERIMENTATION**
with GCP Integration


In [ ]:
#imports 
import sys

sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scripts.plot_utils import (
    set_theme,
    plot_grid,
    plot_barplot,
    plot_barplot_grid,
    plot_histogram,
    plot_numeric_x_numeric_grid,
    plot_numeric_x_across_categories_grid,
    plot_all_numeric_by_base_category_grid,
    plot_categorical_x_categorical_grid,
)
from scripts.gcs_utils import (
    log_dataset_to_gcs,
    log_pipeline_run,
    register_vertex_dataset,
)
from kfp import compiler
from google.cloud import aiplatform

# Components
from vertex.components import (
    load_validate_data,
    split_data,
    oversample_training,
    fit_apply_preprocessing_v1,
    apply_preprocessing_v1,
    train_model,
    evaluate_model,
)

# Pipelines
from vertex.pipelines import (
    preprocessing_pipeline,
    training_pipeline,
    training_pipeline_no_oversample,
)


In [ ]:
# variable declarations
TARGET_COL = "readmission_within_30_days"
ID_COL = "patient_id"
PROJECT_ID = "readmission-543-project"
LOCATION = "us-central1"
BUCKET_ROOT_URI = "gs://readmissions_bucket_v2"


---
## KFP Preprocessing Pipeline

Self-contained KFP v2 components for the preprocessing pipeline.  
Order: `load_validate_data` → `split_data` → `oversample_training` → `fit_apply_preprocessing` → `apply_preprocessing`

In [ ]:
# Components are defined in vertex/components/ and imported above:
#   load_validate_data                          ← ingest.py
#   split_data                                  ← split.py
#   oversample_training                         ← oversample.py
#   fit_apply_preprocessing_v1                  ← preprocessing.py
#   apply_preprocessing_v1                      ← preprocessing.py
#   train_model                                 ← train.py
#   evaluate_model                              ← evaluate.py


In [ ]:
PREPROCESSING_PIPELINE_JSON = "../vertex/pipelines/readmissions_preprocessing_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=preprocessing_pipeline,
    package_path=PREPROCESSING_PIPELINE_JSON,
)

print(f"Pipeline compiled to {PREPROCESSING_PIPELINE_JSON}")


---
## KFP Training Pipeline

`train_model` and `evaluate_model` components extending the preprocessing pipeline.  
Order: `...preprocessing...` → `train_model` → `evaluate_model`

- **`model_type`**: `"logistic"` | `"random_forest"` | `"xgboost"`  
- **`hyperparams_json`**: JSON string of kwargs passed to the chosen estimator (e.g. `'{"n_estimators": 200, "max_depth": 5}'`)

In [ ]:
TRAINING_PIPELINE_JSON = "../vertex/pipelines/readmissions_training_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=training_pipeline,
    package_path=TRAINING_PIPELINE_JSON,
)

print(f"Pipeline compiled to {TRAINING_PIPELINE_JSON}")


---
## Pipeline Submission with Vertex AI Datasets

Register the training CSV as a managed Vertex AI Dataset, then submit the training pipeline.  
The dataset resource name flows through `load_validate_data` metadata → Vertex ML Metadata, giving a native console lineage graph: **Dataset → Pipeline Run → Model**.

- To reuse an existing dataset instead of creating a new one, replace `TabularDataset.create(...)` with `aiplatform.TabularDataset(dataset_name="projects/.../datasets/...")`.

In [ ]:
from pathlib import Path

RAW_TRAIN_PATH = Path("../data/raw/healthcare_readmissions_dataset_train.csv")
DATASET_VERSION = "v0.0"
EXPERIMENT_NAME = "readmissions-model-exp"

log_dataset_to_gcs(
    DATASET_LOCAL_PATH=RAW_TRAIN_PATH,
    VERSION_ID=DATASET_VERSION,
    BUCKET_ROOT_URI=BUCKET_ROOT_URI,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    log_experiment=True,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    description="Raw training data",
    tags=["raw"],
    resume_run=True,
)


In [ ]:
DATASET_VERSION = "v0.0"
dataset, DATASET_GCS_URI = register_vertex_dataset(
    dataset_version=DATASET_VERSION,
    bucket_root_uri=BUCKET_ROOT_URI,
    project_id=PROJECT_ID,
    location=LOCATION,
)
DATASET_RESOURCE_NAME = dataset.resource_name


In [ ]:
# Submit training pipeline, then log the run (with eval metrics) to Vertex Experiments.
MODEL_VERSION = "v0"
EXPERIMENT_NAME = "readmissions-model-exp"

job = aiplatform.PipelineJob(
    display_name=f"readmissions-training-{DATASET_VERSION}-{MODEL_VERSION}",
    template_path=TRAINING_PIPELINE_JSON,
    pipeline_root=f"{BUCKET_ROOT_URI}/pipelines",
    parameter_values={
        "dataset_gcs_uri": DATASET_GCS_URI,
        "dataset_version": DATASET_VERSION,
        "model_type": "xgboost",
        "hyperparams_json": "{}",
    },
)

job.submit()
print(f"Pipeline submitted: {job.display_name}")

log_pipeline_run(
    pipeline_job=job,
    dataset_version=DATASET_VERSION,
    training_dataset_path=DATASET_GCS_URI,
    model_version=MODEL_VERSION,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    wait_for_completion=True,
)


---
## Dataset v1.0 — Outlier Removal

Remove age and BMI outliers identified during EDA (z-score thresholds: age > 3, BMI > 3.5), save the cleaned CSV locally, and register it as a new versioned dataset in GCS and Vertex AI.


In [ ]:
raw_df = pd.read_csv(
    "../data/raw/healthcare_readmissions_dataset_train.csv",
    keep_default_na=False,
    na_values=[""],
)

raw_df.head()

In [ ]:
import scipy.stats as stats
import numpy as np
# age outlier checking
threshold = 3
age_z_scores = np.abs(stats.zscore(raw_df["Age"]))
age_outliers = np.where(age_z_scores > threshold)[0]
print(f"Identified {len(age_outliers)} age outliers at threshold {threshold}:")

display(raw_df.loc[age_outliers, "Age"])


In [ ]:
df_transformed = raw_df.drop(index=age_outliers).reset_index(drop=True)

In [ ]:
# looking into potential bmi outliers

threshold = 3.5
z_scores = np.abs(stats.zscore(df_transformed['BMI']))
bmi_outliers = np.where(z_scores > threshold)[0]
print(f"Identified {len(bmi_outliers)} BMI outliers at threshold {threshold}:")

display(df_transformed.loc[bmi_outliers, 'BMI'])

In [ ]:
df_transformed = df_transformed.copy().drop(index=bmi_outliers).reset_index(drop=True)

In [ ]:
df_transformed.info()

In [ ]:
df_transformed.to_csv("../data/processed/healthcare_readmissions_dataset_train_no_outliers.csv", index=False)

In [ ]:

RAW_TRAIN_PATH = Path("../data/processed/healthcare_readmissions_dataset_train_no_outliers.csv")
DATASET_VERSION = "v1.0"
EXPERIMENT_NAME = "readmissions-model-exp"

log_dataset_to_gcs(
    DATASET_LOCAL_PATH=RAW_TRAIN_PATH,
    VERSION_ID=DATASET_VERSION,
    BUCKET_ROOT_URI=BUCKET_ROOT_URI,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    log_experiment=True,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    description="Training data without outliers",
    tags=["no_outliers"],
    resume_run=True,
)


In [ ]:
DATASET_VERSION = "v1.0"
dataset, DATASET_GCS_URI = register_vertex_dataset(
    dataset_version=DATASET_VERSION,
    bucket_root_uri=BUCKET_ROOT_URI,
    project_id=PROJECT_ID,
    location=LOCATION,
)
DATASET_RESOURCE_NAME = dataset.resource_name


## **XGBoost Experimentation**
- **Baseline**: Default hyperparameters, no oversampling.  
- **Experiment 1**: Default hyperparameters, with oversampling.  
- **Experiment 2**: Hyperparameter tuning with oversampling (e.g. `n_estimators=200`, `max_depth=5`).

### Baseline — XGBoost, No Oversampling

Default XGBoost hyperparameters on the imbalanced training set. Establishes the performance floor before any class-balance intervention.


In [ ]:
NO_OVERSAMPLE_TRAINING_PIPELINE_JSON = "../vertex/pipelines/readmissions_training_no_oversample_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=training_pipeline_no_oversample,
    package_path=NO_OVERSAMPLE_TRAINING_PIPELINE_JSON,
)

print(f"Pipeline compiled to {NO_OVERSAMPLE_TRAINING_PIPELINE_JSON}")


In [ ]:
# Submit training pipeline, then log the run (with eval metrics) to Vertex Experiments.
MODEL_VERSION = "v0"
EXPERIMENT_NAME = "xgboost-exp"

job = aiplatform.PipelineJob(
    display_name=f"readmissions-training-{DATASET_VERSION}-{MODEL_VERSION}-baseline-run-1",
    template_path=NO_OVERSAMPLE_TRAINING_PIPELINE_JSON,
    pipeline_root=f"{BUCKET_ROOT_URI}/pipelines",
    parameter_values={
        "dataset_gcs_uri": DATASET_GCS_URI,
        "dataset_version": DATASET_VERSION,
        "model_type": "xgboost",
        "hyperparams_json": "{}",
    },
)

job.submit()
print(f"Pipeline submitted: {job.display_name}")

log_pipeline_run(
    pipeline_job=job,
    dataset_version=DATASET_VERSION,
    training_dataset_path=DATASET_GCS_URI,
    model_version=MODEL_VERSION,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    wait_for_completion=True,
)


### Experiment 1 — XGBoost with SMOTE Oversampling

Same default hyperparameters as baseline, but the training split is oversampled via SMOTE before fitting. Isolates the effect of class balancing on recall.


In [ ]:
job = aiplatform.PipelineJob(
    display_name=f"readmissions-training-{DATASET_VERSION}-{MODEL_VERSION}-oversampling-experiment-1-run-1",
    template_path=TRAINING_PIPELINE_JSON,
    pipeline_root=f"{BUCKET_ROOT_URI}/pipelines",
    parameter_values={
        "dataset_gcs_uri": DATASET_GCS_URI,
        "dataset_version": DATASET_VERSION,
        "model_type": "xgboost",
        "hyperparams_json": "{}",
    },
)

job.submit()
print(f"Pipeline submitted: {job.display_name}")

log_pipeline_run(
    pipeline_job=job,
    dataset_version=DATASET_VERSION,
    training_dataset_path=DATASET_GCS_URI,
    model_version=MODEL_VERSION,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    wait_for_completion=True,
)

### Experiment 2 — XGBoost + SMOTE, Bayesian Hyperparameter Search (Vertex AI Vizier)

Uses Vertex AI Vizier (Gaussian Process Bandit) instead of exhaustive grid search. Vizier proposes hyperparameter combinations, observes `val_roc_auc` after each trial, and uses that signal to guide subsequent suggestions — finding better configurations with fewer total trials than a full grid search.

- **Search space**: `learning_rate`, `n_estimators`, `max_depth`, `subsample`, `colsample_bytree`
- **Algorithm**: `GAUSSIAN_PROCESS_BANDIT` (Bayesian optimization)
- **Parallelism**: `PARALLEL_TRIAL_COUNT` jobs submitted per round; Vizier updates its model between rounds


In [ ]:
from google.cloud.aiplatform.vizier import Study, pyvizier as vz
from scripts.gcs_utils import _extract_task_metrics
import json
from datetime import datetime, timezone

HPT_EXPERIMENT_NAME = "xgboost-vizier-exp"
DATASET_VERSION = "v1.0"
DATASET_GCS_URI = f"{BUCKET_ROOT_URI}/datasets/readmissions/{DATASET_VERSION}/train.csv"

# Vizier display names must start with a letter and contain only letters, numbers, and underscores
_version_slug = DATASET_VERSION.replace(".", "_").replace("-", "_")
VIZIER_STUDY_DISPLAY_NAME = f"readmissions_xgboost_hpt_{_version_slug}"

# Configure the search space and objective
problem = vz.StudyConfig()
problem.metric_information.append(
    vz.MetricInformation(name="val_roc_auc", goal=vz.ObjectiveMetricGoal.MAXIMIZE)
)
root = problem.search_space.select_root()
root.add_float_param("learning_rate", min_value=0.01, max_value=0.3, scale_type=vz.ScaleType.LOG)
root.add_int_param("n_estimators", min_value=50, max_value=500)
root.add_int_param("max_depth", min_value=2, max_value=8)
root.add_float_param("subsample", min_value=0.6, max_value=1.0)
root.add_float_param("colsample_bytree", min_value=0.6, max_value=1.0)
# ALGORITHM_UNSPECIFIED lets Vertex AI choose the algorithm — defaults to GP Bandit (Bayesian optimization)
problem.algorithm = vz.Algorithm.ALGORITHM_UNSPECIFIED

# create_or_load resumes an existing study if the display name already exists
study = Study.create_or_load(
    display_name=VIZIER_STUDY_DISPLAY_NAME,
    problem=problem,
    project=PROJECT_ID,
    location=LOCATION,
)
print(f"Study: {study.resource_name}")
print(f"  Display name : {VIZIER_STUDY_DISPLAY_NAME}")
print(f"  Algorithm    : ALGORITHM_UNSPECIFIED (Vertex AI defaults to GP Bandit / Bayesian optimization)")
print(f"  Objective    : maximize val_roc_auc")
print(f"  Params       : learning_rate, n_estimators, max_depth, subsample, colsample_bytree")


In [ ]:
MAX_TRIALS = 20
PARALLEL_TRIAL_COUNT = 4   # Vizier suggests N candidates per round; all submitted in parallel

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
completed_count = 0
failed_count = 0

for round_num in range(MAX_TRIALS // PARALLEL_TRIAL_COUNT):
    print(f"\n--- Round {round_num + 1}/{MAX_TRIALS // PARALLEL_TRIAL_COUNT} ({completed_count} done so far) ---")
    trials = study.suggest(count=PARALLEL_TRIAL_COUNT)

    # Submit all trials in this round before waiting on any
    round_jobs = []
    for i, trial in enumerate(trials):
        hyperparams = {name: param.value for name, param in trial.parameters.items()}
        hyperparams["n_estimators"] = int(hyperparams["n_estimators"])
        hyperparams["max_depth"] = int(hyperparams["max_depth"])

        model_version = f"vizier-r{round_num}-t{i}"
        job = aiplatform.PipelineJob(
            display_name=f"readmissions-vizier-{DATASET_VERSION}-{model_version}-{RUN_TIMESTAMP}",
            template_path=TRAINING_PIPELINE_JSON,
            pipeline_root=f"{BUCKET_ROOT_URI}/pipelines",
            parameter_values={
                "dataset_gcs_uri": DATASET_GCS_URI,
                "dataset_version": DATASET_VERSION,
                "model_type": "xgboost",
                "hyperparams_json": json.dumps(hyperparams),
            },
        )
        job.submit()
        round_jobs.append((job, trial, hyperparams, model_version))
        print(f"  Submitted {model_version}: {hyperparams}")

    # Wait for each job, extract val_roc_auc, and report back to Vizier
    for job, trial, hyperparams, model_version in round_jobs:
        try:
            job.wait()
            metrics = _extract_task_metrics(job, task_name="evaluate-model")
            roc_auc = metrics.get("val_roc_auc")
            if roc_auc is None:
                raise ValueError(f"val_roc_auc not in pipeline output. Got: {metrics}")
            measurement = vz.Measurement()
            measurement.metrics["val_roc_auc"] = vz.Metric(value=roc_auc)
            trial.add_measurement(measurement)
            trial.complete()
            completed_count += 1
            print(f"  ✓ {model_version} | val_roc_auc={roc_auc:.4f}")
        except Exception as e:
            trial.complete(infeasible_reason=str(e)[:500])
            failed_count += 1
            print(f"  ✗ {model_version} FAILED: {e}")

print(f"\nSearch complete — {completed_count} succeeded, {failed_count} failed.")


### Vizier Results — Ranked Comparison

List all completed trials from the Vizier study, ranked by `val_roc_auc`. The optimal configuration is printed below the table — use it for the final retrain.


In [ ]:
import pandas as pd

# Inspect the first trial to understand the actual API before parsing all trials
_all_trials = study.trials()
print(f"Total trials: {len(_all_trials)}")

if _all_trials:
    t = _all_trials[0]
    print(f"\nTrial type: {type(t)}")
    print(f"\nPublic attributes (dir):")
    print([a for a in dir(t) if not a.startswith("__")])
    print(f"\nvars / __dict__:")
    try:
        print(vars(t))
    except TypeError:
        print("(not a dict-based object)")
